In [2]:
%pip install python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /Users/ruicongrong/Downloads/mlph-25w-main_2/sheet02/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install gitsource minsearch openai python-dotenv

  Using cached pandas-2.3.3-cp310-cp310-macosx_11_0_arm64.whl.metadata (91 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 22.5 MB/s  0:00:00
Using cached pandas-2.3.3-cp310-cp310-macosx_11_0_arm64.whl (10.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [openai]5;237m━━━ 12/13 [openai]c]

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /Users/ruicongrong/Downloads/mlph-25w-main_2/sheet02/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import json
from openai import OpenAI
from pathlib import Path
from dotenv import load_dotenv


load_dotenv(Path.cwd().parent / ".env", override=True)

client = OpenAI(

    api_key=os.getenv("GROQ_API_KEY"),

    base_url="https://api.groq.com/openai/v1"

)

MODEL = "llama-3.3-70b-versatile"
QUERY = "How does the agentic loop keep calling the model until it stops?"

In [12]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

len_documents = len(documents)
len_documents

72

In [13]:
q1_options = [24, 72, 240, 720]
q1_answer = min(q1_options, key=lambda x: abs(x - len_documents))

print("Q1 raw value:", len_documents)
print("Q1 choose:", q1_answer)

Q1 raw value: 72
Q1 choose: 72


In [14]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

results = index.search(QUERY, num_results=5)

for i, r in enumerate(results, start=1):
    print(i, r["filename"])

1 01-agentic-rag/lessons/14-agentic-loop.md
2 01-agentic-rag/lessons/15-frameworks.md
3 01-agentic-rag/lessons/13-function-calling.md
4 01-agentic-rag/lessons/11-agents-intro.md
5 01-agentic-rag/lessons/16-other-frameworks.md


In [15]:
q2_answer = results[0]["filename"]
print("Q2 choose:", q2_answer)

Q2 choose: 01-agentic-rag/lessons/14-agentic-loop.md


In [16]:

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
""".strip()


PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()


def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(f"filename: {doc['filename']}")
        lines.append(doc["content"])
        lines.append("")

    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    context = build_context(search_results)
    return PROMPT_TEMPLATE.format(question=question, context=context)


def get_input_tokens(response):
    usage = response.usage

    if hasattr(usage, "input_tokens"):
        return usage.input_tokens

    if hasattr(usage, "prompt_tokens"):
        return usage.prompt_tokens

    if isinstance(usage, dict):
        return usage.get("input_tokens") or usage.get("prompt_tokens")

    raise ValueError(f"Cannot read input tokens from usage: {usage}")


def run_rag(question, search_index, num_results=5):
    search_results = search_index.search(question, num_results=num_results)
    prompt = build_prompt(question, search_results)

    input_messages = [
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user", "content": prompt},
    ]

    response = client.responses.create(
        model=MODEL,
        input=input_messages,
    )

    answer = response.output_text
    input_tokens = get_input_tokens(response)

    return answer, input_tokens, search_results

In [17]:
answer_full, tokens_full, search_results_full = run_rag(QUERY, index)

print("Answer:")
print(answer_full)
print()
print("Q3 raw input tokens:", tokens_full)

q3_options = [700, 7000, 70000, 700000]
q3_answer = min(q3_options, key=lambda x: abs(x - tokens_full))

print("Q3 choose:", q3_answer)

Answer:
The agentic loop keeps calling the model until it stops by using a `while` loop that continues to execute as long as the model returns a response with a function call. Here's a step-by-step breakdown of how the loop works:

1. **Initialize the loop**: The loop is initialized with an iteration counter and a flag to track whether the model has returned a function call.
2. **Call the model**: The model is called with the current input, which includes the user's question and any previous messages or tool outputs.
3. **Process the response**: The model's response is processed to check if it contains a function call.
4. **Check for function calls**: If the response contains a function call, the flag is set to `True`, and the function call is executed.
5. **Update the input**: The response output is added to the input for the next iteration, including any tool outputs or messages.
6. **Repeat the loop**: The loop repeats steps 2-5 until the model returns a response without a function 

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len_chunks = len(chunks)
len_chunks

295

In [19]:
q4_options = [70, 295, 1100, 4500]
q4_answer = min(q4_options, key=lambda x: abs(x - len_chunks))

print("Q4 raw value:", len_chunks)
print("Q4 choose:", q4_answer)

Q4 raw value: 295
Q4 choose: 295


In [20]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [21]:
answer_chunked, tokens_chunked, search_results_chunked = run_rag(QUERY, chunk_index)

print("Answer:")
print(answer_chunked)
print()
print("Full-document input tokens:", tokens_full)
print("Chunked input tokens:", tokens_chunked)

ratio = tokens_full / tokens_chunked

print("Ratio full/chunked:", ratio)

Answer:
The agentic loop keeps calling the model until it stops by using a `while True` loop that continues to execute until a certain condition is met. In this case, the condition is that the model returns a response without any function calls.

Here's a step-by-step breakdown of how the loop works:

1. The loop starts with `it = 1`, which is an iteration counter.
2. Inside the loop, the model is called with the current `messages` as input, and the response is stored in the `response` variable.
3. The `messages` list is extended with the output of the model's response.
4. The loop then checks each item in the response's output. If an item is a function call, it is executed, and the output is appended to the `messages` list. The `has_function_calls` flag is set to `True`.
5. If an item is a message, it is printed to the console.
6. The iteration counter `it` is incremented by 1.
7. The loop checks if `has_function_calls` is `False`. If it is, the loop breaks, and the function returns t

In [22]:
q5_options = [
    (1, "about the same"),
    (3, "3x fewer"),
    (10, "10x fewer"),
    (30, "30x fewer"),
]

q5_answer = min(q5_options, key=lambda x: abs(x[0] - ratio))[1]

print("Q5 choose:", q5_answer)

Q5 choose: 3x fewer


In [26]:
import json
import re


def extract_json(text):
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return json.loads(match.group(0))

    raise ValueError(f"Could not parse JSON from model output:\n{text}")


def search_tool(query: str):
    results = chunk_index.search(query, num_results=5)

    output = []
    for r in results:
        output.append({
            "filename": r["filename"],
            "start": r.get("start"),
            "content": r["content"][:1200],
        })

    return output


agent_system_prompt = """
You are a course teaching assistant.

You must choose one action at each step.

Return ONLY valid JSON. No markdown. No explanation outside JSON.

Allowed JSON formats:

1. To search:
{"action": "search", "query": "your search query"}

2. To answer:
{"action": "final", "answer": "your final answer"}

Use search when you need information from the course lesson pages.
Stop and return final when you have enough information.
""".strip()


question = "How does the agentic loop work, and how is it different from plain RAG?"

context_items = []
search_call_count = 0
max_steps = 20

for step in range(max_steps):
    context_text = json.dumps(context_items, ensure_ascii=False, indent=2)

    user_prompt = f"""
Question:
{question}

Current retrieved context:
{context_text}

Decide the next action.
Return only JSON.
""".strip()

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": agent_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=500,
    )

    raw = response.choices[0].message.content
    print(f"\nStep {step + 1} raw output:")
    print(raw)

    action = extract_json(raw)

    if action["action"] == "search":
        query = action["query"]
        print(f"Search call {search_call_count + 1}: {query}")

        result = search_tool(query)
        context_items.append({
            "query": query,
            "results": result,
        })

        search_call_count += 1

    elif action["action"] == "final":
        print("\nFinal answer:")
        print(action["answer"])
        break

    else:
        raise ValueError(f"Unknown action: {action}")

print("\nRaw search call count:", search_call_count)

q6_options = [0, 4, 10, 20]
q6_answer = min(q6_options, key=lambda x: abs(x - search_call_count))

print("Q6 choose:", q6_answer)


Step 1 raw output:
{"action": "search", "query": "agentic loop vs plain RAG"}
Search call 1: agentic loop vs plain RAG

Step 2 raw output:
{"action": "search", "query": "agentic loop vs plain RAG architecture and functionality"}
Search call 2: agentic loop vs plain RAG architecture and functionality

Step 3 raw output:
{"action": "search", "query": "agentic loop architecture and functionality comparison to plain RAG"}
Search call 3: agentic loop architecture and functionality comparison to plain RAG

Step 4 raw output:
{"action": "search", "query": "agentic loop vs plain RAG architecture and functionality details"}
Search call 4: agentic loop vs plain RAG architecture and functionality details

Step 5 raw output:
{"action": "search", "query": "agentic loop vs plain RAG architecture and functionality details comparison"}
Search call 5: agentic loop vs plain RAG architecture and functionality details comparison

Step 6 raw output:
{"action": "search", "query": "agentic loop vs plain RAG

In [27]:
q6_options = [0, 4, 10, 20]
q6_answer = min(q6_options, key=lambda x: abs(x - search_call_count))

print("Q6 choose:", q6_answer)

Q6 choose: 4


In [28]:
print("Submit these choices:")
print("Q1:", q1_answer)
print("Q2:", q2_answer)
print("Q3:", q3_answer)
print("Q4:", q4_answer)
print("Q5:", q5_answer)
print("Q6:", q6_answer)

Submit these choices:
Q1: 72
Q2: 01-agentic-rag/lessons/14-agentic-loop.md
Q3: 7000
Q4: 295
Q5: 3x fewer
Q6: 4
